Overall Accuracy - SVAMP

Normal:
CoT - 80.00
Standard - 78.05
Complex CoT - 74.63

Hypothesis:
CoT - 80.98
Standard - 84.39
Complex CoT - 77.07

In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/SVAMPsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [18]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/h_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/h_CoT_bad.txt'

def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

def process_entry(d):
    """Process a single entry from dev_data."""
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then think step by step through this plan. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step, and correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect" or result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:10<34:41, 10.20s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:10<14:53,  4.40s/it]

Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/205 [00:10<05:51,  1.75s/it]

...

Accuracy: 144 / 180 = 80.00%
Accuracy: 145 / 181 = 80.11%
Accuracy: 146 / 182 = 80.22%
Accuracy: 147 / 183 = 80.33%
Accuracy: 148 / 184 = 80.43%
Accuracy: 149 / 185 = 80.54%
Accuracy: 150 / 186 = 80.65%
Accuracy: 151 / 187 = 80.75%
Accuracy: 152 / 188 = 80.85%
Accuracy: 153 / 189 = 80.95%
Accuracy: 154 / 190 = 81.05%


 95%|█████████▌| 195/205 [02:15<00:02,  4.06it/s]

Accuracy: 155 / 191 = 81.15%
Accuracy: 156 / 192 = 81.25%
Accuracy: 156 / 193 = 80.83%
Accuracy: 157 / 194 = 80.93%
Accuracy: 158 / 195 = 81.03%
Accuracy: 159 / 196 = 81.12%
Accuracy: 160 / 197 = 81.22%


100%|██████████| 205/205 [02:16<00:00,  1.50it/s]

Accuracy: 161 / 198 = 81.31%
Accuracy: 162 / 199 = 81.41%
Accuracy: 163 / 200 = 81.50%
Accuracy: 164 / 201 = 81.59%
Accuracy: 165 / 202 = 81.68%
Accuracy: 165 / 203 = 81.28%
Accuracy: 165 / 204 = 80.88%
Accuracy: 166 / 205 = 80.98%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/h_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/h_Standard_bad.txt'

# === Cleaning Utility ===
def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then answer. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:01<03:42,  1.09s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:02<04:28,  1.32s/it]

Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%
Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%
Accuracy: 9 / 9 = 100.00%
Accuracy: 10 / 10 = 100.00%
Accuracy: 11 / 11 = 100.00%
Accuracy: 12 / 12 = 100.00%
Accuracy: 13 / 13 = 100.00%
Accuracy: 14 / 14 = 100.00%
Accuracy: 14 / 15 = 93.33%
Accuracy: 15 / 16 = 93.75%
Accuracy: 16 / 17 = 94.12%
Accuracy: 17 / 18 = 94.44%


 10%|█         | 21/205 [00:04<00:30,  6.03it/s]

...

Accuracy: 136 / 164 = 82.93%
Accuracy: 136 / 165 = 82.42%
Accuracy: 137 / 166 = 82.53%
Accuracy: 138 / 167 = 82.63%
Accuracy: 139 / 168 = 82.74%
Accuracy: 139 / 169 = 82.25%
Accuracy: 140 / 170 = 82.35%
Accuracy: 141 / 171 = 82.46%


 84%|████████▍ | 172/205 [02:56<01:32,  2.82s/it]

Accuracy: 142 / 172 = 82.56%
Accuracy: 143 / 173 = 82.66%
Accuracy: 144 / 174 = 82.76%
Accuracy: 145 / 175 = 82.86%
Accuracy: 146 / 176 = 82.95%
Accuracy: 147 / 177 = 83.05%
Accuracy: 148 / 178 = 83.15%
Accuracy: 149 / 179 = 83.24%
Accuracy: 150 / 180 = 83.33%
Accuracy: 151 / 181 = 83.43%
Accuracy: 152 / 182 = 83.52%
Accuracy: 153 / 183 = 83.61%
Accuracy: 154 / 184 = 83.70%
Accuracy: 155 / 185 = 83.78%
Accuracy: 156 / 186 = 83.87%
Accuracy: 157 / 187 = 83.96%
Accuracy: 158 / 188 = 84.04%
Accuracy: 159 / 189 = 84.13%
Accuracy: 160 / 190 = 84.21%
Accuracy: 161 / 191 = 84.29%
Accuracy: 162 / 192 = 84.38%
Accuracy: 162 / 193 = 83.94%
Accuracy: 163 / 194 = 84.02%
Accuracy: 164 / 195 = 84.10%
Accuracy: 165 / 196 = 84.18%
Accuracy: 166 / 197 = 84.26%
Accuracy: 167 / 198 = 84.34%
Accuracy: 168 / 199 = 84.42%
Accuracy: 169 / 200 = 84.50%
Accuracy: 170 / 201 = 84.58%
Accuracy: 171 / 202 = 84.65%
Accuracy: 172 / 203 = 84.73%


100%|██████████| 205/205 [02:57<00:00,  1.16it/s]

Accuracy: 172 / 204 = 84.31%
Accuracy: 173 / 205 = 84.39%


91.50 + 2/100 = 92.50, check hypothesis_Standerd

In [17]:
# === Metrics ===
acc = 0
total = 0
error_count = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/SVAMP/h_complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'])  # Ground truth

        # === Hypothesis + Complex CCoT Prompt ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Finish your response with: the answer is <answer>."
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: the answer is <answer>."
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None
            error_count += 1

        # === Structured Logging ===
        log_block = (
            f'Q: {q}\n'
            f'RESPONSE:\n{ans_model}\n'
            f'EXTRACTED:\n{extracted}\n'
            f'GROUND_TRUTH:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ INCORRECT OR INVALID\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

# === Final Summary ===
summary = f"\n✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)

  0%|          | 1/205 [00:02<08:13,  2.42s/it]

Accuracy: 0 / 1 = 0.00%


  1%|          | 2/205 [00:05<08:49,  2.61s/it]

Accuracy: 0 / 2 = 0.00%


  1%|▏         | 3/205 [00:07<08:00,  2.38s/it]

...

Accuracy: 156 / 203 = 76.85%


100%|█████████▉| 204/205 [08:40<00:02,  2.58s/it]

Accuracy: 157 / 204 = 76.96%


100%|██████████| 205/205 [08:42<00:00,  2.55s/it]

Accuracy: 158 / 205 = 77.07%

✅ Accuracy: 158 / 205 = 77.07%
❌ Errors: 2

